# Faza 2 — Puna naracija „Šta je AI" (OpenAudio S1-mini)

Generiše celu naraciju tvojim kloniranim glasom. **Prvi put** skida model (zaključan → treba token) i **kešira ga na Google Drive**; svaki sledeći put ga samo **kopira sa Drive-a** (bez skidanja).

## Jednokratno podešavanje (uradi PRE prvog pokretanja)
1. Napravi besplatan HuggingFace nalog: https://huggingface.co/join
2. Otvori https://huggingface.co/fishaudio/openaudio-s1-mini i klikni **Agree and access repository**.
3. Napravi token (Read): https://huggingface.co/settings/tokens → *New token* → kopiraj (`hf_...`).
4. U Colab-u, leva traka → 🔑 **Secrets** → *Add new secret* → Name: **`HF_TOKEN`**, Value: tvoj token, uključi **Notebook access**.

Posle ovoga token je tu zauvek — ne unosiš ga više.

**Pre pokretanja:** Runtime → Change runtime type → **GPU**. Pa **Runtime → Run all**.

> Zašto se instalacija (korak 2) ponavlja svaku sesiju: Colab runtime je privremen i obriše se kad ga ugasiš. Model NE skidamo ponovo jer ga čuvamo na Drive-u (korak 3).

## 1. GPU + Google Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive; drive.mount('/content/drive')

## 2. Instalacija fish-speech (~3 min; ponavlja se svaku sesiju jer se VM briše)

In [ ]:
%cd /content
!git clone https://github.com/fishaudio/fish-speech.git 2>/dev/null || echo '(repo vec postoji)'
%cd /content/fish-speech
!apt-get -qq install -y portaudio19-dev
!pip install -e . -q
!pip install -q torchvision==0.23.0 "transformers==4.57.3"
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Upali GPU: Runtime -> Change runtime type -> GPU, pa Restart.'

## 3. Model: kopiraj sa Drive-a, ili skini jednom (pa keširaj na Drive)

Prvi put traži `HF_TOKEN` iz Secrets i skida ~3.6 GB, pa kopira na Drive. Sledeći put samo kopira sa Drive-a — bez tokena i bez skidanja.

In [ ]:
import os, shutil, subprocess
LOCAL = 'checkpoints/openaudio-s1-mini'
DRIVE = '/content/drive/MyDrive/ai-glas/openaudio-s1-mini'
MODEL_DIR = LOCAL
CODEC = f'{LOCAL}/codec.pth'
os.makedirs('checkpoints', exist_ok=True)

if os.path.exists(f'{DRIVE}/codec.pth'):
    print('Model nadjen na Drive-u -> kopiram lokalno (bez skidanja)...')
    if os.path.exists(LOCAL): shutil.rmtree(LOCAL)
    shutil.copytree(DRIVE, LOCAL, ignore=shutil.ignore_patterns('.cache'))
else:
    print('Nema na Drive-u -> skidam sa HuggingFace (treba HF_TOKEN u Secrets)...')
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    subprocess.run(['huggingface-cli','download','fishaudio/openaudio-s1-mini','--local-dir',LOCAL], check=True)
    print('Cuvam model na Drive za sledeci put (~par min)...')
    os.makedirs(os.path.dirname(DRIVE), exist_ok=True)
    if os.path.exists(DRIVE): shutil.rmtree(DRIVE)
    shutil.copytree(LOCAL, DRIVE, ignore=shutil.ignore_patterns('.cache'))

assert os.path.exists(CODEC), f'NEMA {CODEC} — proveri da si prihvatio uslove modela na HF i da je HF_TOKEN u Secrets.'
print('Model spreman:', [f for f in os.listdir(LOCAL) if not f.startswith('.')])

## 4. Parametri (referenca, transkript, naracija)

`PROMPT_TEXT` mora da odgovara prvih ~22s tvog uzorka. `SHIFTS=[0.0]` = samo trenutni glas (prerušavanje dodajemo kasnije ako odlučiš).

In [ ]:
REF_FULL = '/content/drive/MyDrive/ai-glas/owner-sample.wav'
DRIVE_OUT = '/content/drive/MyDrive/ai-glas'
SHIFTS = [0.0]   # prvo SAMO trenutni glas; kasnije dodaj npr. -0.5 za prerusavanje

PROMPT_TEXT = "Veštačka inteligencija danas više nije nešto što gledamo samo u filmovima. Ona piše tekst, pravi slike, pomaže u programiranju i odgovara na pitanja iz skoro svake oblasti."

NARRATION = """Veštačka inteligencija u 2026. više nije naučna fantastika. Danas ti piše kod, pravi slike i odgovara na pitanja bolje nego ikad. Za nekih sedam minuta objasniću ti šta ona zaista jeste, šta sve može i kako da je koristiš potpuno besplatno.
Ako se prvi put susrećeš sa ovim svetom, opusti se. Sve ćemo proći redom, jednostavnim rečima, bez nepotrebnih komplikacija.
Krenimo od osnovnog pitanja: šta je zapravo veštačka inteligencija? Kada danas kažemo AI, najčešće mislimo na takozvane velike jezičke modele. To su programi koji su trenirani na ogromnoj količini teksta i koda. Iz svega što su pročitali, oni nauče da predvide koja reč najverovatnije dolazi sledeća.
Zvuči jednostavno, ali baš iz tog predviđanja sledeće reči izranja nešto moćno. Model može da napiše ceo tekst, da odgovori na pitanje ili da reši zadatak. Važno je da zapamtiš jedno: model ne razmišlja kao čovek. On veoma dobro pogađa na osnovu obrazaca koje je naučio.
Hajde da vidimo to u praksi. Otvorim alat, ukucam pitanje običnim jezikom, i za par sekundi dobijem jasan odgovor. Isto tako mogu da tražim da mi napiše kod, da mi skrati dugačak tekst ili da mi objasni pojam koji ne razumem.
A ko pravi te modele? U 2026. tri imena se najčešće pominju. Anthropic, koji stoji iza Claude-a. OpenAI, poznat po GPT i Codex modelima. I Google, sa porodicom modela koja se zove Gemini. Svaki od njih ima svoje jače i slabije strane, pa se isplati probati više njih.
Dve stvari su se drastično promenile u poslednje dve godine. Prvo, modeli sada mogu da obrade mnogo više teksta odjednom. Drugo, cena korišćenja je znatno pala. To zajedno znači da je danas moćan alat dostupan skoro svakome, a ne samo velikim firmama.
Ali budimo pošteni i oko mana. Model ponekad zvuči potpuno sigurno, a zapravo greši. To zovemo halucinacija. Zato uvek proveri važne informacije i ne veruj svemu na reč. Alat je tu da ti pomogne, a ne da razmišlja umesto tebe.
Kako da počneš već danas, i to besplatno? Dovoljno je da otvoriš jedan od besplatnih alata u pregledaču i da mu postaviš prvo pitanje. Najbolji savet koji mogu da ti dam: budi jasan i konkretan. Što tačnije opišeš šta želiš, to je bolji rezultat.
Ako ti je ovo bilo korisno, prijavi se na kanal. U sledećem videu pokazujem ti kako da napišeš prompt koji stvarno daje dobre rezultate.
Hvala ti što si gledao. Vidimo se u sledećem videu."""
print('Parametri spremni.')

## 5. BRZA provera (jedna rečenica) — da ne čekaš dugu naraciju ako nešto ne valja

Ako je `SANITY RC: 0` i čuješ glas → sve radi, idi na korak 6. Ako `RC: 1`, pošalji mi ispis.

In [ ]:
import librosa, soundfile as sf, subprocess, IPython.display as ipd
y, sr = librosa.load(REF_FULL, sr=None, mono=True); sf.write('/content/ref_chk.wav', y[:int(22*sr)], sr)
subprocess.run(['python','fish_speech/models/dac/inference.py','-i','/content/ref_chk.wav','--checkpoint-path',CODEC], check=True)
r = subprocess.run(['python','fish_speech/models/text2semantic/inference.py','--text','Ovo je kratak test glasa.','--prompt-text',PROMPT_TEXT,'--prompt-tokens','fake.npy','--checkpoint-path',MODEL_DIR,'--num-samples','1'], capture_output=True, text=True)
print('SANITY RC:', r.returncode)
if r.returncode != 0:
    print(r.stderr[-1500:])
else:
    subprocess.run(['python','fish_speech/models/dac/inference.py','-i','codes_0.npy','--checkpoint-path',CODEC], check=True)
    print('OK — okruzenje radi!'); ipd.display(ipd.Audio('fake.wav'))

## 6. Puna naracija (~3–4 min zvuka; potraje)

Za svaki pomak iz `SHIFTS`: pomeri referencu → enkoduj → generiši iz cele naracije → dekoduj → snimi na Drive i pusti.

In [ ]:
import librosa, soundfile as sf, subprocess, shutil, IPython.display as ipd
y, sr = librosa.load(REF_FULL, sr=None, mono=True); y = y[:int(22*sr)]
res = []
for s in SHIFTS:
    print(f'\n===== POMAK {s} =====')
    yy = y if s == 0 else librosa.effects.pitch_shift(y, sr=sr, n_steps=s)
    refp = f'/content/ref_{s}.wav'; sf.write(refp, yy, sr)
    subprocess.run(['python','fish_speech/models/dac/inference.py','-i',refp,'--checkpoint-path',CODEC], check=True)
    subprocess.run(['python','fish_speech/models/text2semantic/inference.py','--text',NARRATION,'--prompt-text',PROMPT_TEXT,'--prompt-tokens','fake.npy','--checkpoint-path',MODEL_DIR,'--num-samples','1'], check=True)
    subprocess.run(['python','fish_speech/models/dac/inference.py','-i','codes_0.npy','--checkpoint-path',CODEC], check=True)
    outp = f'{DRIVE_OUT}/narration_sta-je-ai_shift_{s}.wav'; shutil.copy('fake.wav', outp); res.append((s, outp))
    print('Snimljeno:', outp)
print('\n===== PRESLUSAJ =====')
for s, outp in res:
    print(f'--- pomak {s} ---'); ipd.display(ipd.Audio(outp))